# Inspire AI — Test Suite (Phase J skeleton)

This notebook exercises the Inspire-AI agent in isolation with deterministic inputs
and asserts structural properties of its outputs. It is industry-agnostic: no
specific schema, domain, or column is hardcoded.

## Cells

1. Imports + fixtures
2. UseCaseBundle round-trip (dict ↔ bundle)
3. VALIDATOR_REGISTRY contract
4. NEXT_ACTIONS add/snapshot/summary
5. Phase Q1 grounding (hermetic)
6. Phase Q2 canonical dedup
7. Phase L final-gate
8. 10 negative cases (see §4.11 in TESTING_GUIDE.md)

In [0]:
# Cell 1: imports + fixtures
import sys, os
sys.path.insert(0, os.path.abspath('../agent'))
# The main notebook's mega-cell must be executed first in the test runner.
# In Databricks this is done via dbutils.notebook.run or %run.

def _assert(cond, msg):
    if not cond:
        raise AssertionError(msg)
    print(f'  PASS: {msg}')

print('Test suite bootstrapping...')

In [0]:
# Cell 2: UseCaseBundle round-trip
sample = {
    'No': '1', 'Name': 'Sample Use Case',
    'Business Domain': 'SampleDomain', 'Subdomain': 'SampleSub',
    'type': 'BUSINESS', 'Analytics Technique': 'Descriptive',
    'Priority': 'Medium', 'Primary Table': 'sample_table',
    'Statement': 'This is a sample use-case statement of sufficient length.',
    'Solution': 'This is a sample solution description of sufficient length.',
    'Business Value': 'Improves X by Y%',
    'extra_field': 'preserved',
}
b = UseCaseBundle.from_dict(sample)
_assert(b.name == 'Sample Use Case', 'from_dict preserves name')
_assert(b.provenance.get('extra_field') == 'preserved', 'from_dict preserves unknowns')
d2 = b.to_dict()
_assert(d2['Name'] == 'Sample Use Case', 'to_dict round-trips Name')
_assert(d2['extra_field'] == 'preserved', 'to_dict round-trips provenance')

In [0]:
# Cell 3: VALIDATOR_REGISTRY contract
_assert('has_primary_table' in VALIDATOR_REGISTRY, 'has_primary_table registered')
_assert('statement_min_length' in VALIDATOR_REGISTRY, 'statement_min_length registered')
results = run_validators(b)
_assert(all(p for p, _ in results.values()), 'sample bundle passes all validators')
# Negative: empty bundle
empty = UseCaseBundle()
nr = run_validators(empty)
_assert(not nr['has_primary_table'][0], 'empty bundle fails primary_table')
_assert(not nr['statement_min_length'][0], 'empty bundle fails statement_min_length')

In [0]:
# Cell 4: NEXT_ACTIONS scaffold
before = len(NEXT_ACTIONS.snapshot())
NEXT_ACTIONS.add('TEST_RULE', severity='INFO', phase='J', step='UnitTest',
                 evidence='unit test emission')
after = len(NEXT_ACTIONS.snapshot())
_assert(after == before + 1, 'add grows snapshot')
summary = NEXT_ACTIONS.summary()
_assert(summary['total'] == after, 'summary total matches snapshot len')

In [0]:
# Cell 5: 13_generate_genie_code_for widget DSL parsing
def _parse_ggc(raw):
    s = (raw or "5").strip().lower()
    if s == "all":
        return "ALL"
    try:
        n = int(s)
        return max(0, n)
    except ValueError:
        return None

_assert(_parse_ggc("all") == "ALL", "'all' parses to ALL sentinel")
_assert(_parse_ggc("ALL") == "ALL", "'ALL' is case-insensitive")
_assert(_parse_ggc("5") == 5, "'5' parses to int 5")
_assert(_parse_ggc("0") == 0, "'0' parses to disabled")
_assert(_parse_ggc("") == 5, "empty defaults to 5")
_assert(_parse_ggc("-3") == 0, "negatives clamp to 0")
_assert(_parse_ggc("garbage") is None, "garbage rejected")

In [0]:
# Cell 6: Phase Q1 grounding (hermetic) - skipped in skeleton; requires instance
print('Cell 6: Q1 grounding test requires DatabricksInspire instance - skipped in skeleton')

In [0]:
# Cell 7: Phase Q2 canonical dedup - skipped in skeleton
print('Cell 7: Q2 canonical dedup test requires DatabricksInspire instance - skipped')

In [0]:
# Cell 8: Phase L final-gate - skipped in skeleton
print('Cell 8: Phase L test requires DatabricksInspire instance - skipped')

In [0]:
# Cell 9: 10 negative cases (structural, industry-agnostic)
neg_cases = [
    {},
    {'Name': ''},
    {'Primary Table': ''},
    {'Name': 'x', 'Statement': 'short'},
    {'Name': 'x', 'Solution': 'short'},
    {'Name': 'x', 'Primary Table': 'tbl', 'Statement': 'ok'*20, 'Solution': 'ok'*20,
     'Analytics Technique': ''},
    {'Name': 'dup', 'Primary Table': 'tbl'},
    {'Name': 'dup', 'Primary Table': 'tbl'},
    {'Business Domain': 'Other'},
    {'Business Domain': 'Unknown'},
]
for i, nc in enumerate(neg_cases):
    b = UseCaseBundle.from_dict(nc)
    r = run_validators(b)
    fails = [k for k, (ok, _) in r.items() if not ok]
    print(f'  neg {i}: {len(fails)} validator failures')
    _assert(len(fails) > 0, f'neg case {i} has at least one failure')
print('All negative cases asserted.')